In [2]:
import pandas as pd

In [3]:
df_aps = pd.read_csv("https://minio.lab.sspcloud.fr/juleslapeuf/df_aps.csv")

In [4]:
df_aps.head()

,event_id,tournament,uniquetournament_id,season,season_id,round,bestOfSets,bestOfLegs,home_score,away_score,...,HighestCheckout_away,CheckoutsOver100_home,CheckoutsOver100_away,CheckoutsAccuracy_home,CheckoutsAccuracy_away,CheckoutsAccuracy_homeTotal,CheckoutsAccuracy_awayTotal,date,R_home_new,R_away_new
0,15142767,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,1.0,3.0,...,62.0,1.0,0.0,5.0,10.0,13.0,25.0,2025-12-11 19:15:00,0.326050,0.002538
1,15142749,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,3.0,0.0,...,72.0,0.0,0.0,9.0,3.0,25.0,10.0,2025-12-11 20:12:00,0.857104,-0.049318
2,15142718,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,3.0,0.0,...,130.0,2.0,1.0,9.0,5.0,14.0,11.0,2025-12-11 21:04:00,1.719075,-0.058423
3,15142764,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,3.0,1.0,...,92.0,0.0,0.0,11.0,8.0,35.0,22.0,2025-12-11 21:59:00,0.107832,-0.138110
4,15142751,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,3.0,0.0,...,100.0,1.0,1.0,9.0,4.0,25.0,14.0,2025-12-12 12:49:00,0.206039,-0.189061


In [5]:
df_aps.columns

Index(['event_id', 'tournament', 'uniquetournament_id', 'season', 'season_id',
       'round', 'bestOfSets', 'bestOfLegs', 'home_score', 'away_score',
       'home_id', 'away_id', 'home_name', 'away_name', 'Average3Darts_home',
       'Average3Darts_away', 'Thrown180_home', 'Thrown180_away',
       'ThrownOver140_home', 'ThrownOver140_away', 'ThrownOver100_home',
       'ThrownOver100_away', 'HighestCheckout_home', 'HighestCheckout_away',
       'CheckoutsOver100_home', 'CheckoutsOver100_away',
       'CheckoutsAccuracy_home', 'CheckoutsAccuracy_away',
       'CheckoutsAccuracy_homeTotal', 'CheckoutsAccuracy_awayTotal', 'date',
       'R_home_new', 'R_away_new'],
      dtype='str')

Ce jeu de données est un ensemble de matchs de fléchettes (un ligne = un match) :
- event_id : identifiant unique de chaque match
- tournamnent : nom du tournoi dans lequel le match s'est joué (avec uniquetournament_id son code)
- season : la saison du tournoi (season_id étant l'identifiant)
- round : le niveau du match dans le tournoi (finale, demi-finale...)
- bestOfSets : le match s'est joué au meilleur des ... sets
- bestOfLegs : les sets se sont joués au meilleur des ... manches
- home_score/away_score : nb de sets de chaque joueurs
- home_id/away_id / home_name/away_name : identifiant unique et et nom des joueurs
- date : la date exacte (à la minute) du match

Les autres colonnes correspondent aux statistiques du matchs. Pour donner un peu de contexte, pour une manche, les deux joueurs partent avec un total de 501 et ils enlèvent de leur total la somme des points des 3 fléchettes et cela ainsi de suite jusqu'à soustraire exactement 501 au total. Le premier qui soustrait 501 remporte la manche.
Le maximum de points possibles pour un lancer de trois fléchettes est 180 points, c'est-à-dire les 3 fléchettes dans la case triple 20. Il est impossible de terminer par une fléchette triple, donc le plus haut checkout (c'est-à-dire fin de manche) est 170, à savoir deux triple 20 et le rond du milieu (bulleseye) pour 50. Bref, c'est du détail, mais ça permet de comprendre un peu les ordres de grandeur.

- Average3Darts : La moyenne des 3 fléchettes sur l'ensemble du match
- Thrown180 : Nombre de lancers à 180
- ThrownOver140/Over100 : Nombre de lancers au-delà de 140, de 100
- HighestCheckout : La fin de manche la plus élevée (max 170)
- CheckoutsOver100 : Nombre de fin de manche au-dessus de 100
- CheckoutsAccuracy : Réussite des fins de mancher (nb de fois où le joueur a terminé la manche quand il le pouvait)

Enfin, j'ai calculé à partir d'un score de type ELO (connu aux échecs notamment) un rating pour chaque joueur qui évolue dans le temps

- R_home_new : rating du joueur home
- R_away_new : rating du joueur away

In [8]:
# Nombre d'apparitions de chaque joueur
player_counts = pd.concat([
    df_aps["home_id"],
    df_aps["away_id"]
]).value_counts()

# Joueurs apparaissant au moins 4 fois
eligible_players = player_counts[player_counts >= 4].index

# Conservation des matchs entre joueurs éligibles
df_aps_filtered = df_aps[
    df_aps["home_id"].isin(eligible_players)
    & df_aps["away_id"].isin(eligible_players)
].reset_index(drop=True).copy()

In [9]:
len(df_aps_filtered)

3656

In [10]:
stats = [
    "Average3Darts",
    "Thrown180",
    "ThrownOver140",
    "ThrownOver100",
    "HighestCheckout",
    "CheckoutsOver100",
    "CheckoutsAccuracy",
]

home = df_aps[["event_id", "date", "home_id"] + [f"{s}_home" for s in stats]].copy()
home = home.rename(columns={"home_id": "player_id"})
home.columns = ["event_id", "date", "player_id"] + stats

away = df_aps[["event_id", "date", "away_id"] + [f"{s}_away" for s in stats]].copy()
away = away.rename(columns={"away_id": "player_id"})
away.columns = ["event_id", "date", "player_id"] + stats

df_players = pd.concat([home, away], ignore_index=True)
df_players = df_players.sort_values(["player_id", "date", "event_id"])

# Moyenne des matchs précédents, sans inclure le match courant
df_moyennes = df_players.copy()

df_moyennes[stats] = (
    df_moyennes
    .groupby("player_id")[stats]
    .transform(lambda col: col.shift().expanding().mean())
)

df_moyennes = df_moyennes.reset_index(drop=True)

In [11]:
df_moyennes.head()

,event_id,date,player_id,Average3Darts,Thrown180,ThrownOver140,ThrownOver100,HighestCheckout,CheckoutsOver100,CheckoutsAccuracy
0,10863191,2022-11-25 15:05:00,26086,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10914577,2022-12-21 19:15:00,26086,0.0,1.000000,4.000000,13.000000,118.0,1.0,4.00
2,10914596,2022-12-29 14:30:00,26086,0.0,3.000000,9.000000,22.500000,121.0,1.0,8.00
3,11098566,2023-03-04 15:20:00,26086,0.0,2.666667,10.333333,22.666667,126.0,1.0,7.00
4,12121151,2024-03-02 21:05:00,26086,0.0,3.000000,10.500000,22.750000,127.0,1.0,7.25
